# Self-RAG에서 영감 받은 근거 검사

초안의 근거 점수를 평가하고 기준 미만일 때 한 번 수정합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 이 절은 답변 초안을 만든 뒤 근거 문서와 얼마나 맞는지 점수를 요청하고, 기준 미만이면 한 번 수정합니다. 생성, 평가, 수정이 서로 다른 단계이며 평가 프롬프트와 기준값도 실행 기록의 일부입니다.

**다른 상황에 적용.** 보고서 요약에서 숫자와 날짜가 원문에 있는지 평가하고 근거가 약하면 해당 문장을 삭제해 다시 생성할 수 있습니다. 이때 평가 모델이 ‘맞다’고 했다는 사실과 실제 원문 검증 결과를 구분해야 합니다.

**비교와 경계.** 원래 Self-RAG는 검색·생성 과정에 반성 토큰을 통합한 연구 방법입니다. 이 절은 생성 후 근거 검사에서 영감을 받은 축소 흐름입니다. judge 점수는 사실의 증명이 아니므로 코드 규칙과 사람 평가로 교차 확인합니다.


## 실행 환경 준비

고정한 Python 패키지를 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3
import sys
print("준비 완료:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
준비 완료: 3.12.13


## .env 찾기

가까운 .env에서 키를 읽고 존재 여부만 표시합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 현재 폴더에서 가까운 .env를 찾되 비밀 값은 출력하지 않습니다.
import os, time
from dotenv import find_dotenv, load_dotenv
env_path=find_dotenv(usecwd=True)
load_dotenv(env_path)
print("OPENAI_API_KEY:","있음" if os.getenv("OPENAI_API_KEY") else "없음")

OPENAI_API_KEY: 있음


## 공통 데이터

Ch12의 축소 데이터와 Ch11 방식의 정답 문서 ID를 사용합니다.

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 배포 묶음에서도 같은 상대 경로를 찾도록 프로젝트 루트를 거슬러 올라갑니다.
import json
from pathlib import Path
import numpy as np
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"labs").is_dir())
data_dir=root/"labs/part4/ch14-rag-patterns/data"
docs=json.loads((data_dir/"documents.json").read_text(encoding="utf-8"))
queries=json.loads((data_dir/"queries.json").read_text(encoding="utf-8"))
print("문서·질문:",len(docs),len(queries))

문서·질문: 12 8


## 모델과 임베딩

모델과 256차원 문서 벡터를 준비합니다.

**이 셀의 코드 읽기.** `OpenAIEmbeddings(...)`는 문자열을 수치 벡터로 변환할 임베딩 클라이언트를 준비합니다. 실제 호출은 `embed_query()`나 `embed_documents()`에서 일어납니다. `embed_documents(texts)`는 여러 문장을 입력받아 문장 수만큼의 벡터를 반환합니다. `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
# 256차원은 Ch12와 같은 비교 조건이며 행 하나가 문서 하나입니다.
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
chat=init_chat_model("gpt-4.1-mini",model_provider="openai",temperature=0)
embedder=OpenAIEmbeddings(model="text-embedding-3-small",dimensions=256)
vectors=np.array(embedder.embed_documents([d["text"] for d in docs]))
print("임베딩 배열:",vectors.shape)

임베딩 배열: (12, 256)


## 검색·생성 함수

검색 점수, 토큰, 시간, 추정 비용을 같은 방식으로 기록합니다.

**이 셀의 코드 읽기.** `search(question: str, k: int = 3)`, `ask(prompt: str)`, `show_hits(hits: list[tuple[dict[str, Any], float]])`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `embed_query(text)`는 검색 질문 한 개를 벡터 하나로 반환합니다.


In [5]:
# 코사인 유사도는 내적을 두 벡터의 길이로 나누어 계산합니다.
from typing import Any
usage_log: list[dict[str, float | int]] = []

def search(question: str, k: int = 3) -> list[tuple[dict[str, Any], float]]:
    q=np.array(embedder.embed_query(question))
    scores=vectors@q/(np.linalg.norm(vectors,axis=1)*np.linalg.norm(q))
    # argsort 결과를 뒤집어 점수가 큰 문서부터 k개를 고릅니다.
    return [(docs[i],float(scores[i])) for i in np.argsort(scores)[::-1][:k]]
def ask(prompt: str) -> tuple[str, int, float]:
    started=time.perf_counter()
    msg=chat.invoke(prompt)
    usage=msg.usage_metadata or {}
    details=usage.get("input_token_details", {})
    input_tokens=usage.get("input_tokens", 0)
    cached_tokens=details.get("cache_read", 0)
    output_tokens=usage.get("output_tokens", 0)
    tokens=usage.get("total_tokens", 0)
    # 2026-09-10 gpt-4.1-mini 공개 단가로 추정하며 실제 청구액과 다를 수 있습니다.
    usd=((input_tokens-cached_tokens)*0.40+cached_tokens*0.10+output_tokens*1.60)/1_000_000
    usage_log.append({"input":input_tokens,"cached":cached_tokens,"output":output_tokens,"usd":usd})
    return msg.text,tokens,round((time.perf_counter()-started)*1000,1)

def show_hits(hits: list[tuple[dict[str, Any], float]]) -> list[tuple[str, float]]:
    return [(d["doc_id"],round(score,3)) for d,score in hits]

## 검색하고 초안 생성

전자책 규정을 검색해 첫 답을 생성합니다. 대출 권수·기간과 인용 ID가 원문에 있는지 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [6]:
# 검색된 전자책 규정만 전달해 출처가 있는 첫 답을 만듭니다.
question="전자책은 몇 권까지 며칠 동안 빌릴 수 있나요?"
hits=search(question,k=2)
context="\n".join(f"[{d['doc_id']}] {d['text']}" for d,_ in hits)
draft,gen_tokens,gen_ms=ask(f"근거로 답하고 [문서ID]를 쓰세요.\n{context}\n질문:{question}")
print("초안:",draft)

초안: 전자책은 동시에 세 권까지 대출할 수 있고, 대출 기간은 7일입니다. [LIB-004]


## 근거 충실도 평가

초안에 통제 오류 문장을 붙여 근거 점수를 요청합니다. 숫자를 0~1로 제한한 뒤 0.5가 나온 과정을 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [7]:
# 원문에 없는 문장을 넣은 검사 초안으로 낮은 점수 경로를 확인합니다.
# 수정 분기를 확인하도록 원문에 없는 연체료 문장을 고정 초안에 추가합니다.
import re
draft_for_check=draft+" 연체료는 500원입니다."
score_text,judge_tokens,judge_ms=ask(f"근거로 확인되는 답의 비율을 0~1 숫자 하나로 쓰세요.\n근거:{context}\n답:{draft_for_check}")
match=re.search(r"(?:0(?:\.\d+)?|1(?:\.0+)?)",score_text)
score=max(0.0,min(1.0,float(match.group()) if match else 0.0))
threshold=0.9
print("검사 초안:",draft_for_check)
print("근거 점수:",score,"기준:",threshold)

검사 초안: 전자책은 동시에 세 권까지 대출할 수 있고, 대출 기간은 7일입니다. [LIB-004] 연체료는 500원입니다.
근거 점수: 0.5 기준: 0.9


## 한 번만 수정

점수가 0.9 미만이면 원문에 없는 문장을 한 번 제거합니다. 최종 답과 세 호출의 합계 사용량을 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [8]:
# 점수가 기준 미만이면 근거 없는 문장을 한 번 제거합니다.
if score<threshold:
    final,revise_tokens,revise_ms=ask(f"근거 없는 내용을 빼고 답을 다시 쓰세요.\n근거:{context}\n답:{draft_for_check}")
else: final,revise_tokens,revise_ms=draft,0,0.0
print("최종:",final)
print("전체 토큰·ms:",gen_tokens+judge_tokens+revise_tokens,gen_ms+judge_ms+revise_ms)

최종: 전자책은 동시에 세 권까지 대출할 수 있고, 대출 기간은 7일입니다. [LIB-004]
전체 토큰·ms: 409 2873.3
